In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from typing import List
from functools import partial
import torch
torch.autograd.set_detect_anomaly(True)
from PIL import Image
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from transformers import ProcessorMixin, MllamaProcessor, AutoTokenizer, AutoImageProcessor
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from datasets import load_dataset, Dataset

from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.segmentation import SemanticSegmentationTool

from torchmetrics import JaccardIndex

from src.kitti_tracking import KittiDataset
from src.kitti_tracking_hf import KittiHFIterableDataset
from arc_trainer import ArcTrainer
from arc_utils import ARCCollator, ArcProcessor, ExtendedLMHead, ExtendEmbedding

/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [3]:
root_dir = "/mnt/ssd/kitti_tracking"
n_steps, n_pred_steps = 1, 0

dataset_builder = KittiHFIterableDataset(
	root_dir=root_dir,
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)
dataset = dataset_builder.to_hf_dataset()

In [4]:
model_id = "meta-llama/Llama-3.2-11B-Vision"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
processor = MllamaProcessor.from_pretrained(model_id)

vq_enc: Encoder = load_model(f"./checkpoints/encoder.pkl", "cpu")
vq_enc.eval()
for param in vq_enc.parameters(): param.requires_grad = False

collator = ARCCollator(
    processor, processor.tokenizer,
    vq_enc
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.12it/s]


In [5]:
model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        # 'down_proj', 'gate_proj', 'up_proj',
        # "embed_tokens", "lm_head",
    ],
    use_dora=True, # optional DoRA 
    init_lora_weights="gaussian"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# llm_processor = AutoProcessor.from_pretrained(model_id)

trainable params: 12,410,880 || all params: 10,655,352,355 || trainable%: 0.1165


In [ ]:
# model.enable_
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

training_args = TrainingArguments(
	max_steps=3000,
	output_dir='./results',
	logging_dir='./logs',
	gradient_checkpointing=True,
    # gradient_checkpointing_kwargs={'use_reentrant': False},
	per_device_train_batch_size=1,
	per_device_eval_batch_size=1,
	# num_train_epochs=1,
	# logging_steps=10,
	# save_total_limit=2,
	# disable_tqdm=False,       # enable progress bar
	logging_strategy="steps",
	logging_steps=10,          # show every step
	logging_first_step=True,  # show step 0/1
	bf16=True,
	# use_cpu=True,
	remove_unused_columns=False,
	learning_rate=1e-6,
	lr_scheduler_type="cosine",
	max_grad_norm=1,
	gradient_accumulation_steps=8,
)
arc_trainer = Trainer(
	# ckpt_path="",
	model=model,
	args=training_args,
	train_dataset=dataset,
	data_collator=partial(collator, prompts=['people', 'vehicles']),
)
arc_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
arc_trainer.save_model("./final_model")

In [ ]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, param.grad.norm())

In [ ]:
# VQ-VAE
def preprocess(img: Image.Image) -> torch.Tensor:
	img = torch.unsqueeze(T.ToTensor()(img), 0)
	return map_pixels(img)  # (1 - 2 * 0.1) * x + 0.1

def vq_encode(image, model, dev):
	x = preprocess(image).to(dev)
	z_logits = model(x.to(dev))
	z = torch.argmax(z_logits, axis=1)
	
	return z

def vq_decode(codes, model):
	z = F.one_hot(codes, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()

	x_stats = model(z).float()
	x_rec = unmap_pixels(torch.sigmoid(x_stats[:, :3]))
	x_rec = T.ToPILImage(mode='RGB')(x_rec[0])

	return x_rec

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
enc: Encoder = load_model("./checkpoints/encoder.pkl", device)
dec: Decoder = load_model("./checkpoints/decoder.pkl", device)

In [ ]:
sem_segm_tool = SemanticSegmentationTool(model_name="CIDAS/clipseg-rd64-refined")

In [ ]:
# load "codewords" from VAE
new_tokens = [
    f"<vq_{i}>" for i in range(enc.blocks[-1].conv.w.shape[0])
] + ["<begin_mask>", "<end_mask>"]

processor.tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(processor.tokenizer))

In [ ]:
# load dataset for fine-tuning
root_dir = "C:/Users/ngoak/data/kitti_tracking"
n_steps, n_pred_steps = 16, 3


train_ds = KittiDataset(
	root_dir, "training",
	n_steps, n_pred_steps,
	inp_transforms=T.Compose([
		T.Resize((240, 640)),
	])
)
for sample_dict in train_ds:
	rgb_images, depth_images = sample_dict['rgb'], sample_dict['depth']
	
	# rgb_code =
	# rgb_code_str =
	break

In [ ]:
iou_metric = JaccardIndex(task="binary")
tgt_class = ["person"]
prompt = "Goal"
for image in rgb_images:
	_prompt = "<|image|>"

	sem_mask = sem_segm_tool(image, tgt_class)
	z = vq_encode(sem_mask.convert("RGB"), enc, device)
	_z = z[0].cpu().numpy()
	mask_s = "<begin_mask>" + "".join([
		f"<vq_{_z[i, j]}>"
		for i in range(_z.shape[0])
		for j in range(_z.shape[1])
	]) + "<end_mask>"

	_sem_mask = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	patch = Image.new("RGB", (50, 50), (255, 255, 255))
	sem_mask.paste(patch, (100, 100))
	_sem_mask2 = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	iou = 1 - iou_metric(_sem_mask2, _sem_mask)
	
	_prompt = "<|image|>" +\
	f"Thought: To ensure novel information is included in the transmission, I generate mask that reduces curiosity.\n" +\
	f"Action: {mask_s}\n" +\
	f"Observation: {iou:6f}\n" 

	print(iou)

	break

In [ ]:
gt = "<|image1|>" +\
    f"<|begin_of_text|> The curious mask that marks regions that might posibly contain {','.join(tgt_class)}"# +\
    # f"{mask_s}\n"

gt

In [ ]:
_prompt

In [ ]:
sem_mask

In [ ]:
sem_mask

In [ ]:
x = preprocess(rgb_images[0]).to(model.device)
z_logits = enc(x.to(device))
z = torch.argmax(z_logits, axis=1)
z.shape
display(T.ToPILImage(mode='RGB')(x[0]))

In [ ]:
_z = z[0].cpu().numpy()
mask_s = "<begin_mask>" + "".join([
    f"<vq_{_z[i, j]}>"
    for i in range(_z.shape[0])
    for j in range(_z.shape[1])
]) + "<end_mask>"


In [ ]:
processor.tokenizer(mask_s, return_tensors="pt", ).input_ids

In [ ]:
z = vq_encode(rgb_images[0], enc, device)

x_rec = vq_decode(z, dec)

display(x_rec)

In [ ]:
import requests


url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# prompt = "<|image|><|begin_of_text|>If I had to write a haiku for this one"
# inputs = processor(image, prompt, return_tensors="pt").to(model.device)

# output = model.generate(**inputs, max_new_tokens=30)

In [ ]:
def preprocess_fn(samples, clip_segm, vq_enc, ):
	
	def vq_preprocess(img: Image.Image) -> torch.Tensor:
		img = torch.unsqueeze(T.ToTensor()(img), 0)
		return map_pixels(img)  # (1 - 2 * 0.1) * x + 0.1
	
	images = sample["rgb"]   # list of PIL or arrays
	prompts = ["object"]     # or your actual prompts

	processed = processor(
		prompts=prompts,
		images=images,
		return_tensors="pt",
		padding=True,
		truncation=True
	)
	_pixel_values = processed.get("pixel_values", None)

	# labels for causal LM
	labels = processed["input_ids"].clone()

	# ignore padding in loss
	labels[labels == processor.tokenizer.pad_token_id] = -100

	return {
		"input_ids": processed["input_ids"][0].clone().detach(),
		"attention_mask": processed["attention_mask"][0].clone().detach(),
		"pixel_values": _pixel_values[0].clone().detach() if _pixel_values is not None else None,
		"labels": labels.clone().detach(),  # causal LM
		"aspect_ratio_ids": processed["aspect_ratio_ids"][0].clone().detach(),  # causal LM
		"aspect_ratio_mask": processed["aspect_ratio_mask"][0].clone().detach(),  # causal LM
	}